# 10 - Foundry IQ through Azure API Management

## Required permissions

| Function | Required permission |
|---|---|
| Read existing sources and create the knowledge base | **Search Service Contributor** plus **Search Index Data Reader** (or **Search Index Data Contributor**) on Search |
| Configure the gateway, broker app and shared Web IQ source | **API Management Service Contributor** or equivalent gateway configuration access; permission to manage the broker app and approve its delegated consent; **Search Service Contributor** on Search |
| Retrieve through APIM | Tenant-approved access to the broker app's delegated **`user_impersonation`** scope and access to the indexed SharePoint documents |
| Gateway access to Search | APIM managed identity: **Search Index Data Reader** on Search |

**Resource names:** keep notebook 09's `WORKSHOP_PREFIX`, for example `alice01` (3-20 lowercase letters/digits, starting with a letter). This notebook creates only `alice01-apim-kb` and reuses `alice01-kb-blob` / `alice01-kb-sharepoint`. The Web IQ source (`WEBIQ_APIM_SOURCE_NAME`) is shared and read-only in this notebook; it must match the APIM policy. Shared gateway, broker app and services are unchanged. Prefixes prevent naming collisions, not enforce access isolation.

Notebook 09 calls the knowledge base **directly** and stores the Web IQ key in its MCP knowledge source for notebook and portal retrieval. This notebook reuses the Blob and SharePoint sources but selects a separate, keyless Web IQ source. Retrieval goes **through an APIM gateway**, which supplies the key and forwards the signed-in user's Search context using on-behalf-of exchange.

```text
notebook --(Entra token for the broker app)--> APIM /iq
                                                 | validate-jwt
                                                 | on-behalf-of: same user -> Azure AI Search token
                                                 | add Web IQ key from an APIM secret
                                                 | call Search as the APIM managed identity
                                                 v
                                        knowledge base --> Blob + SharePoint + Web IQ
```

What changes for the caller:

| | Notebook 09 (direct) | Notebook 10 (APIM) |
|---|---|---|
| Caller's token | Azure AI Search | the **broker app** (a different app registration) |
| Web IQ key | stored in the Web IQ knowledge source | only in APIM |
| SharePoint permissions | caller's Search token | same user, via the on-behalf-of exchange |
| Caller needs Search RBAC for retrieval | yes | no, only the broker token |

The knowledge base combines notebook 09's Blob and SharePoint sources with the gateway's shared Web IQ source. Source lookup and knowledge-base creation still require direct Search access; only retrieval goes through APIM.

## Prerequisites

- Notebook 09 completed with `CLEAN_UP = False`, so `{prefix}-kb-blob` and `{prefix}-kb-sharepoint` exist.
- Deployed workshop resources and an **existing APIM service**, identified by `APIM_RESOURCE_ID` in `.env`. Bicep does not create APIM.
- **Required backend access:** the APIM service's **system-assigned managed identity** must have **Search Index Data Reader on the Azure AI Search service** used by the gateway. Enabling the identity alone is insufficient. **The setup scripts do not grant this role.** Without backend data access, retrieval can return **HTTP 403**, even with a valid broker token and a successful on-behalf-of exchange.
- Completed gateway and broker setup using `scripts\setup-broker-app.ps1` and `scripts\setup-apim-policy.ps1`, including approved delegated consent.
- `BROKER_APP_CLIENT_ID`, `APIM_GATEWAY_URL` and `WEBIQ_APIM_SOURCE_NAME` in `.env`, using the values printed by the setup scripts.
- An active `az login` session in the deployment tenant.

### Grant the required Search role manually

1. On the existing APIM resource, enable **Identity > System assigned** if it is not already enabled.
2. Open the **Azure AI Search resource** targeted by the gateway, then **Access control (IAM) > Add role assignment**.
3. Select **Search Index Data Reader**. For the member, select the APIM service's **system-assigned managed identity**, not the broker app or the signed-in user.
4. Save the assignment at the **Search service scope** and allow time for role propagation before retrying retrieval. Creating this assignment requires role-assignment permissions at that scope.

A role on the APIM resource or Foundry project does not supply this Search data access. The gateway still forwards the caller's Search context separately; this backend grant does not change SharePoint document permissions.

In [ ]:
import base64
import json
import os
import re
from pathlib import Path
from uuid import UUID

import requests
from azure.identity import AzureCliCredential, get_bearer_token_provider
from dotenv import load_dotenv

root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
if not (root / 'infra' / 'main.bicep').is_file():
    raise RuntimeError('Open from the workshop root or notebooks directory.')
load_dotenv(root / '.env', override=True)
load_dotenv(root / '.env.azure', override=True)
required = ('AZURE_TENANT_ID', 'AZURE_SEARCH_ENDPOINT', 'BROKER_APP_CLIENT_ID', 'APIM_GATEWAY_URL')
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise RuntimeError(f'Missing settings: {missing}. Complete gateway setup and see .env.example.')
tenant_id = str(UUID(os.environ['AZURE_TENANT_ID']))
broker_client_id = str(UUID(os.environ['BROKER_APP_CLIENT_ID']))
prefix = os.getenv('WORKSHOP_PREFIX') or input('Resource prefix (e.g. alice01): ').strip()
if not re.fullmatch(r'[a-z][a-z0-9]{2,19}', prefix):
    raise ValueError('Use 3-20 lowercase letters/digits, starting with a letter.')

search_endpoint = os.environ['AZURE_SEARCH_ENDPOINT'].rstrip('/')
apim_url = os.environ['APIM_GATEWAY_URL'].rstrip('/')
api_version = '2026-08-01-preview'

blob_source_name = f'{prefix}-kb-blob'
sharepoint_source_name = f'{prefix}-kb-sharepoint'
webiq_source_name = os.getenv('WEBIQ_APIM_SOURCE_NAME', 'iq-workshop-webiq')
knowledge_base_name = f'{prefix}-apim-kb'

credential = AzureCliCredential(tenant_id=tenant_id)
search_token = get_bearer_token_provider(credential, 'https://search.azure.com/.default')
broker_token = get_bearer_token_provider(credential, f'api://{broker_client_id}/.default')
print('Gateway:', apim_url)
print('Knowledge base:', knowledge_base_name)

RuntimeError: Missing settings: ['APIM_GATEWAY_URL']. Complete gateway setup and see .env.example.

## 1. Get a token for the broker app

The caller requests a token for the **broker app registration**, containing its approved `user_impersonation` permission. With `AzureCliCredential`, request **`api://<broker-client-id>/.default`**: the SDK converts this to the resource `api://<broker-client-id>` for Azure CLI. Passing `/user_impersonation` directly incorrectly includes the permission name in the resource URL and causes **AADSTS500011** before token decoding.

The cell decodes the token to inspect its audience and delegated scope. APIM performs the actual token validation.

### Manual broker-app configuration

In Entra **App registrations**, select the broker app identified by `BROKER_APP_CLIENT_ID`:

1. Under **Expose an API**, set the Application ID URI to `api://<broker-client-id>` and enable a delegated scope named `user_impersonation`.
2. Under **Authorized client applications**, add Azure CLI (`04b07795-8ddb-461a-bbee-02f9e1bf7b46`) and select that exposed scope.
3. Set `api.requestedAccessTokenVersion` to **2** in the app manifest. The app and its enterprise application must exist in the configured deployment tenant.
4. For the later APIM on-behalf-of exchange, add the **Azure Cognitive Search delegated `user_impersonation` permission**, obtain the required consent, and configure the broker credential plus `broker-client-id` and `tenant-id` named values in APIM. `setup-broker-app.ps1` automates these steps.

`/.default` is the SDK token-request syntax, not a new scope to create under Expose an API. Do not create an Application ID URI ending in `/user_impersonation` or follow a login suggestion that appends `/.default` to that permission name. After correcting the configuration cell, rerun it before the token-inspection cell.

In [ ]:
claims = json.loads(base64.urlsafe_b64decode(broker_token().split('.')[1] + '==='))
print('audience (aud):', claims['aud'])
print('scope (scp):   ', claims.get('scp'))
print('user:          ', claims.get('preferred_username') or claims.get('upn'))
if claims['aud'] not in (broker_client_id, f'api://{broker_client_id}'):
    raise RuntimeError('The token is not for the broker app.')
if 'user_impersonation' not in claims.get('scp', '').split():
    raise RuntimeError('The broker token is missing user_impersonation. Check the exposed scope, Azure CLI pre-authorization and consent.')

## 2. Check the shared Web IQ source and create the knowledge base

The Blob and SharePoint sources already exist from notebook 09. `setup-apim-policy.ps1` creates the **shared** Web IQ source, named in the APIM policy (`WEBIQ_APIM_SOURCE_NAME`). This notebook only reads that source and does not overwrite it. Unlike notebook 09's direct source, the gateway source stores no key; APIM adds the key as per-source control headers on every request.

These setup calls go directly to Search using the signed-in user's token. Only *retrieval* goes through APIM.

In [ ]:
response = requests.get(
    f'{search_endpoint}/knowledgesources/{webiq_source_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    timeout=(10, 120),
)
if response.status_code == 404:
    raise RuntimeError('Run setup-apim-policy.ps1 to create the shared Web IQ source first. This notebook does not create or overwrite it.')
response.raise_for_status()
webiq_source = response.json()
if webiq_source.get('kind') != 'mcpServer' or webiq_source.get('mcpServerParameters', {}).get('serverURL') != 'https://api.microsoft.ai/v3/mcp':
    raise RuntimeError('The shared source is not the expected Web IQ MCP source. Check WEBIQ_APIM_SOURCE_NAME and gateway setup.')
print('Shared Web IQ knowledge source found (read-only):', webiq_source_name)

In [ ]:
source_kinds = {}
for source_name in (blob_source_name, sharepoint_source_name):
    response = requests.get(
        f'{search_endpoint}/knowledgesources/{source_name}?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    if response.status_code == 404:
        raise RuntimeError(f'{source_name} not found. Complete notebook 09 first (CLEAN_UP = False).')
    response.raise_for_status()
    source_kinds[source_name] = response.json()['kind']
source_kinds[webiq_source_name] = 'mcpServer'
print('Source kinds:', source_kinds)

# Reuse notebook 09's model and source-routing instructions.
response = requests.get(
    f'{search_endpoint}/knowledgebases/{prefix}-kb?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
)
if response.status_code == 404:
    raise RuntimeError(f'{prefix}-kb not found. Complete notebook 09 first (CLEAN_UP = False).')
response.raise_for_status()
base_kb = response.json()

knowledge_base = {
    'name': knowledge_base_name,
    'description': 'Workshop policies (Blob and SharePoint) plus Web IQ, reached through APIM.',
    'retrievalInstructions': base_kb['retrievalInstructions'].replace(f'{prefix}-kb-webiq', webiq_source_name),
    'answerInstructions': (
        (base_kb.get('answerInstructions') or '') + ' Answer from the retrieved evidence, not from memory. '
        'Do not claim external access is unavailable when web results were retrieved. '
        'If evidence is insufficient, explain what it does not establish; do not offer to fetch sources later '
        'or ask permission to search, because retrieval has already run. '
        'Do not infer the latest release from an incomplete list of older versions.'
    ),
    'outputMode': 'answerSynthesis',
    'knowledgeSources': [{'name': name} for name in source_kinds],
    'models': base_kb['models'],
    'retrievalReasoningEffort': {'kind': 'low'},
}
response = requests.put(
    f'{search_endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}',
    headers={'Authorization': f'Bearer {search_token()}'},
    json=knowledge_base, timeout=(10, 90),
)
response.raise_for_status()
print('Knowledge base ready:', knowledge_base_name)

## 3. Ask through the gateway

Look at the request headers: only the broker token. There is no Web IQ key, no `x-ms-query-source-authorization` and no Search token. APIM supplies all of them. The URL is the APIM gateway (`/iq/...`), not the Search endpoint.

Start with an internal question. If retrieval returns **403**, verify the required **Search Index Data Reader** assignment for APIM's managed identity in the prerequisites before changing broker-token scopes or Web IQ authentication.

In [ ]:
question = 'What is our policy for booking international travel?'
response = requests.post(
    f'{apim_url}/iq/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={'Authorization': f'Bearer {broker_token()}'},
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'includeActivity': True,
        'maxRuntimeInSeconds': 60,
        'knowledgeSourceParams': [
            {'knowledgeSourceName': blob_source_name, 'kind': source_kinds[blob_source_name],
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': sharepoint_source_name, 'kind': source_kinds[sharepoint_source_name],
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer',
             'includeReferences': True, 'includeReferenceSourceData': True},
        ],
    }, timeout=(10, 120),
)
response.raise_for_status()
result = response.json()
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text'))
print('\nReferences:')
for reference in result.get('references', []):
    source_data = reference.get('sourceData') or {}
    if reference.get('type') == 'mcpServer':
        try:
            parsed = json.loads(source_data.get('content') or '{}')
            if isinstance(parsed, dict):
                source_data = {**source_data, **parsed}
        except (TypeError, ValueError):
            pass
    print('-', reference.get('type'), reference.get('id'),
          source_data.get('url') or source_data.get('webUrl') or reference.get('docKey') or '')
print('\nActivity:')
for step in result.get('activity', []):
    print('-', step.get('type'), step.get('knowledgeSourceName', ''), step.get('count', ''), step.get('error') or '')
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity output above.')

The next question needs current public information. The Web IQ key is added by APIM, so the notebook never touches it. Web reference source data is requested and its URL is printed beside the reference ID.

An MCP activity count of 1 means one processed result, not necessarily enough evidence to establish the latest release. Inspect `result['references']` for the retrieved content. If retrieval succeeded, an answer claiming no external access is a synthesis problem, not evidence of missing authentication.

In [30]:
question = 'What is the latest stable release of Python, and when was it released?'
response = requests.post(
    f'{apim_url}/iq/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={'Authorization': f'Bearer {broker_token()}'},
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'includeActivity': True,
        'maxRuntimeInSeconds': 60,
        'knowledgeSourceParams': [
            {'knowledgeSourceName': blob_source_name, 'kind': source_kinds[blob_source_name],
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': sharepoint_source_name, 'kind': source_kinds[sharepoint_source_name],
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer',
             'includeReferences': True, 'includeReferenceSourceData': True},
        ],
    }, timeout=(10, 120),
)
response.raise_for_status()
result = response.json()
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text'))
print('\nReferences:')
for reference in result.get('references', []):
    source_data = reference.get('sourceData') or {}
    if reference.get('type') == 'mcpServer':
        try:
            parsed = json.loads(source_data.get('content') or '{}')
            if isinstance(parsed, dict):
                source_data = {**source_data, **parsed}
        except (TypeError, ValueError):
            pass
    print('-', reference.get('type'), reference.get('id'),
          source_data.get('url') or source_data.get('webUrl') or reference.get('docKey') or '')
print('\nActivity:')
for step in result.get('activity', []):
    print('-', step.get('type'), step.get('knowledgeSourceName', ''), step.get('count', ''), step.get('error') or '')
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity output above.')

The latest stable Python release is Python 3.15.2, released on October 25, 2026 [ref_id:0][ref_id:0].

References:
- mcpServer 0 https://www.python.org/downloads/?keys=latest

Activity:
- modelQueryPlanning   
- mcpServer iq-workshop-webiq 1 
- modelAnswerSynthesis   
- agenticReasoning   


Finally a question that needs both: internal policy and the outside world.

In [ ]:
question = 'Does our travel policy meet current public guidance on business travel? Compare them.'
response = requests.post(
    f'{apim_url}/iq/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}',
    headers={'Authorization': f'Bearer {broker_token()}'},
    json={
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'includeActivity': True,
        'maxRuntimeInSeconds': 90,
        'knowledgeSourceParams': [
            {'knowledgeSourceName': blob_source_name, 'kind': source_kinds[blob_source_name],
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': sharepoint_source_name, 'kind': source_kinds[sharepoint_source_name],
             'includeReferences': True, 'includeReferenceSourceData': True},
            {'knowledgeSourceName': webiq_source_name, 'kind': 'mcpServer',
             'includeReferences': True, 'includeReferenceSourceData': True},
        ],
    }, timeout=(10, 150),
)
response.raise_for_status()
result = response.json()
for message in result.get('response', []):
    for part in message.get('content', []):
        print(part.get('text'))
print('\nReferences:')
for reference in result.get('references', []):
    source_data = reference.get('sourceData') or {}
    if reference.get('type') == 'mcpServer':
        try:
            parsed = json.loads(source_data.get('content') or '{}')
            if isinstance(parsed, dict):
                source_data = {**source_data, **parsed}
        except (TypeError, ValueError):
            pass
    print('-', reference.get('type'), reference.get('id'),
          source_data.get('url') or source_data.get('webUrl') or reference.get('docKey') or '')
print('\nActivity:')
for step in result.get('activity', []):
    print('-', step.get('type'), step.get('knowledgeSourceName', ''), step.get('count', ''), step.get('error') or '')
if response.status_code == 206 or any(step.get('error') for step in result.get('activity', [])):
    raise RuntimeError('Retrieval was partial or a source failed. Inspect the activity output above.')

## 4. Check that the gateway rejects the wrong callers

APIM's `validate-jwt` policy accepts only tokens for the broker app with the `user_impersonation` scope. Two calls that should fail with **401**:

1. no token at all
2. a valid Azure AI Search token (right user, wrong audience)

In [ ]:
url = f'{apim_url}/iq/knowledgebases/{knowledge_base_name}/retrieve?api-version={api_version}'
body = {'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': 'hello'}]}]}

no_token = requests.post(url, json=body, timeout=(10, 60))
print('No token:', no_token.status_code)

wrong_audience = requests.post(
    url, headers={'Authorization': f'Bearer {search_token()}'}, json=body, timeout=(10, 60),
)
print('Search token (wrong audience):', wrong_audience.status_code)

if no_token.status_code != 401 or wrong_audience.status_code != 401:
    raise RuntimeError('The gateway should reject both calls with 401. Check the APIM policy.')

## Cleanup

Deletes only this notebook's knowledge base. The shared Web IQ source is retained, along with the gateway, broker app and notebook 09's Blob and SharePoint sources. Notebook 09 has separate opt-in cleanup for its knowledge resources.

In [ ]:
CLEAN_UP = False
if CLEAN_UP:
    response = requests.delete(
        f'{search_endpoint}/knowledgebases/{knowledge_base_name}?api-version={api_version}',
        headers={'Authorization': f'Bearer {search_token()}'}, timeout=(10, 90),
    )
    if response.status_code != 404:
        response.raise_for_status()
    print('Deleted or already absent:', knowledge_base_name)
else:
    print('Cleanup disabled.')

## References

- [Foundry IQ and agentic retrieval](https://learn.microsoft.com/azure/search/agentic-retrieval-overview)
- [Per-user access with `x-ms-query-source-authorization`](https://learn.microsoft.com/azure/search/search-document-level-access-overview)
- [Microsoft identity platform: on-behalf-of flow](https://learn.microsoft.com/entra/identity-platform/v2-oauth2-on-behalf-of-flow)
- [APIM `validate-jwt` policy](https://learn.microsoft.com/azure/api-management/validate-jwt-policy)
- [APIM `authentication-managed-identity` policy](https://learn.microsoft.com/azure/api-management/authentication-managed-identity-policy)